# Rede Neural MLP - PyTorch
## Previsão de Churn - Telco Customer

**Objetivo:** construir, treinar e avaliar uma MLP (Multi-Layer Perceptron) em PyTorch para classificação de churn, comparando com os baselines.

**Arquitetura:** MLP com 2 camadas ocultas + Early Stopping + Batching

**Este notebook vale 25% da nota do Tech Challenge.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os
import mlflow
import mlflow.pytorch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# ===== SEED GLOBAL PARA REPRODUTIBILIDADE =====
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Bibliotecas carregadas!')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')
print(f'Seed: {SEED}')

## 1. Carregar e Preparar Dados

In [ ]:
# Carregar dados processados
df = pd.read_csv('../data/processed/telco_churn_clean.csv')

# Target
TARGET = 'Churn'
df[TARGET] = df[TARGET].map({'Yes': 1, 'No': 0})

# Separar X e y
X = df.drop(TARGET, axis=1)
y = df[TARGET]

# One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

# Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Escalonamento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Guardar numero de features
INPUT_DIM = X_train_scaled.shape[1]

print(f'Features: {INPUT_DIM}')
print(f'Treino: {X_train_scaled.shape[0]} amostras')
print(f'Teste:  {X_test_scaled.shape[0]} amostras')
print(f'Churn no treino: {y_train.mean():.2%}')
print(f'Churn no teste:  {y_test.mean():.2%}')

## 2. Preparar Dados para PyTorch
### 2.1 Dataset Customizado

In [ ]:
class ChurnDataset(Dataset):
    """
    Dataset customizado para carregar dados de churn no PyTorch.
    Converte arrays numpy para tensores.
    """
    def __init__(self, X, y):
        self.X = torch.FloatTensor(np.array(X))
        self.y = torch.FloatTensor(np.array(y))
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Criar datasets
BATCH_SIZE = 64

train_dataset = ChurnDataset(X_train_scaled, y_train)
test_dataset = ChurnDataset(X_test_scaled, y_test)

# Criar DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Batch size: {BATCH_SIZE}')
print(f'Batches de treino: {len(train_loader)}')
print(f'Batches de teste: {len(test_loader)}')
print()
# Verificar um batch
X_batch, y_batch = next(iter(train_loader))
print(f'Shape de um batch X: {X_batch.shape}')
print(f'Shape de um batch y: {y_batch.shape}')

## 3. Definir Arquitetura da MLP

In [ ]:
class ChurnMLP(nn.Module):
    """
    MLP para classificacao binaria de churn.
    
    Arquitetura:
    - Camada 1: input_dim -> 64 (BatchNorm + ReLU + Dropout)
    - Camada 2: 64 -> 32 (BatchNorm + ReLU + Dropout)
    - Saida: 32 -> 1 (logit para BCEWithLogitsLoss)
    """
    def __init__(self, input_dim):
        super(ChurnMLP, self).__init__()
        
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        self.layer2 = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        self.output = nn.Linear(32, 1)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.output(x)
        return x

# Instanciar modelo
model = ChurnMLP(INPUT_DIM).to(device)

# Contar parametros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('=== Arquitetura da MLP ===')
print(model)
print()
print(f'Total de parametros: {total_params:,}')
print(f'Parametros treinaveis: {trainable_params:,}')

## 4. Configurar Treinamento

In [ ]:
# Loss function: BCEWithLogitsLoss (ja inclui sigmoid internamente)
criterion = nn.BCEWithLogitsLoss()

# Otimizador: Adam
LEARNING_RATE = 0.001
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Configuracoes de treinamento
NUM_EPOCHS = 100
PATIENCE = 10  # Early stopping

print('=== Configuracao do Treinamento ===')
print(f'Loss: BCEWithLogitsLoss')
print(f'Optimizer: Adam (lr={LEARNING_RATE})')
print(f'Epochs: {NUM_EPOCHS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Early stopping patience: {PATIENCE}')
print(f'Device: {device}')

## 5. Loop de Treinamento com Early Stopping

In [ ]:
# Listas para armazenar historico
train_losses = []
val_losses = []

# Early stopping
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None
best_epoch = 0

print('Iniciando treinamento...')
print()

for epoch in range(NUM_EPOCHS):
    # ===== TREINO =====
    model.train()
    train_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        # Forward
        outputs = model(X_batch).squeeze()
        loss = criterion(outputs, y_batch)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # ===== VALIDACAO =====
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()
    
    val_loss /= len(test_loader)
    val_losses.append(val_loss)
    
    # ===== EARLY STOPPING =====
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        best_epoch = epoch + 1
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Print a cada 10 epochs
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS} | '
              f'Train Loss: {train_loss:.4f} | '
              f'Val Loss: {val_loss:.4f} | '
              f'Patience: {patience_counter}/{PATIENCE}')
    
    # Parar se patience esgotou
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping na epoch {epoch+1}!')
        break

# Carregar melhor modelo
model.load_state_dict(best_model_state)
total_epochs = len(train_losses)

print()
print(f'Treinamento concluido!')
print(f'Melhor epoch: {best_epoch}')
print(f'Melhor val loss: {best_val_loss:.4f}')
print(f'Total de epochs: {total_epochs}')

## 6. Visualizar Treinamento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

epochs_range = range(1, total_epochs + 1)

# Grafico 1: Loss curves completas
axes[0].plot(epochs_range, train_losses, label='Train Loss', color='#3498db', linewidth=2)
axes[0].plot(epochs_range, val_losses, label='Val Loss', color='#e74c3c', linewidth=2)
axes[0].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Melhor epoch ({best_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Curvas de Loss - Treino vs Validacao', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Grafico 2: Diferenca entre train e val (detectar overfitting)
diff = [v - t for t, v in zip(train_losses, val_losses)]
axes[1].plot(epochs_range, diff, color='#9b59b6', linewidth=2)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Melhor epoch ({best_epoch})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val Loss - Train Loss')
axes[1].set_title('Gap entre Validacao e Treino (Overfitting check)', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Se o gap aumenta muito, indica overfitting.')
print(f'Early stopping ajuda a parar no ponto ideal.')

## 7. Avaliar MLP

In [ ]:
# Avaliar no conjunto de teste
model.eval()
all_preds = []
all_probs = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch).squeeze()
        probs = torch.sigmoid(outputs).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_targets.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_targets = np.array(all_targets)

# Metricas
acc = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds)
rec = recall_score(all_targets, all_preds)
f1 = f1_score(all_targets, all_preds)
roc_auc = roc_auc_score(all_targets, all_probs)

print('=== Resultados da MLP ===')
print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1-score:  {f1:.4f}')
print(f'ROC-AUC:   {roc_auc:.4f}')
print()
print(classification_report(all_targets, all_preds, target_names=['No Churn', 'Churn']))

# Matriz de confusao
cm = confusion_matrix(all_targets, all_preds)
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
ax.set_xlabel('Previsto')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusao - MLP PyTorch')
plt.tight_layout()
plt.show()

# Salvar resultado
result_mlp = {
    'model_name': 'MLP_PyTorch',
    'accuracy': round(acc, 4),
    'precision': round(prec, 4),
    'recall': round(rec, 4),
    'f1_score': round(f1, 4),
    'roc_auc': round(roc_auc, 4)
}

## 8. Comparação: MLP vs Baselines

In [ ]:
# Carregar resultados dos baselines
df_baselines = pd.read_csv('../models/baseline_results.csv', index_col=0)

# Adicionar MLP
mlp_row = pd.DataFrame([result_mlp]).set_index('model_name')
df_all = pd.concat([df_baselines, mlp_row])

print('=== COMPARACAO COMPLETA ===')
print()
print(df_all.to_string())
print()

# Grafico comparativo
metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
x = np.arange(len(metrics))
width = 0.2

fig, ax = plt.subplots(figsize=(16, 6))

models = df_all.index.tolist()
colors = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c']

for i, (model_name, color) in enumerate(zip(models, colors)):
    values = df_all.loc[model_name, metrics].values
    bars = ax.bar(x + i * width, values, width, label=model_name, color=color)
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=7)

ax.set_ylabel('Score')
ax.set_title('Comparacao: MLP vs Baselines', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([m.replace('_', ' ').title() for m in metrics])
ax.legend()
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.show()

# Analise
best_model = df_all['roc_auc'].idxmax()
best_roc = df_all['roc_auc'].max()
print(f'\nMelhor modelo por ROC-AUC: {best_model} ({best_roc:.4f})')

## 9. Análise de Trade-off (Falso Positivo vs Falso Negativo)

In [ ]:
# Testar diferentes thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
threshold_results = []

for t in thresholds:
    preds_t = (all_probs >= t).astype(int)
    threshold_results.append({
        'threshold': t,
        'precision': precision_score(all_targets, preds_t),
        'recall': recall_score(all_targets, preds_t),
        'f1_score': f1_score(all_targets, preds_t)
    })

df_thresh = pd.DataFrame(threshold_results)

print('=== Analise por Threshold ===')
print(df_thresh.to_string(index=False))
print()

# Grafico
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_thresh['threshold'], df_thresh['precision'], 'o-', label='Precision', color='#3498db', linewidth=2)
ax.plot(df_thresh['threshold'], df_thresh['recall'], 's-', label='Recall', color='#e74c3c', linewidth=2)
ax.plot(df_thresh['threshold'], df_thresh['f1_score'], '^-', label='F1-score', color='#2ecc71', linewidth=2)
ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Trade-off: Precision vs Recall por Threshold', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Interpretacao:')
print('- Threshold BAIXO (0.3): mais recall (detecta mais churn), menos precision (mais alarmes falsos)')
print('- Threshold ALTO (0.7): mais precision (menos alarmes falsos), menos recall (perde clientes)')
print('- Para CHURN: recall e mais importante (nao queremos perder clientes)')
print('- Recomendacao: threshold entre 0.4-0.5 para equilibrar deteccao e custo de acao')

## 10. Registro no MLflow

In [ ]:
# Registrar MLP no MLflow
mlflow.set_experiment('churn-mlp')

with mlflow.start_run(run_name='MLP_PyTorch'):
    # Parametros
    mlflow.log_param('model_type', 'MLP_PyTorch')
    mlflow.log_param('architecture', 'input->64->32->1')
    mlflow.log_param('activation', 'ReLU')
    mlflow.log_param('dropout', 0.3)
    mlflow.log_param('learning_rate', LEARNING_RATE)
    mlflow.log_param('batch_size', BATCH_SIZE)
    mlflow.log_param('max_epochs', NUM_EPOCHS)
    mlflow.log_param('patience', PATIENCE)
    mlflow.log_param('best_epoch', best_epoch)
    mlflow.log_param('total_epochs', total_epochs)
    mlflow.log_param('optimizer', 'Adam')
    mlflow.log_param('loss_function', 'BCEWithLogitsLoss')
    mlflow.log_param('random_state', SEED)
    
    # Metricas
    mlflow.log_metric('accuracy', result_mlp['accuracy'])
    mlflow.log_metric('precision', result_mlp['precision'])
    mlflow.log_metric('recall', result_mlp['recall'])
    mlflow.log_metric('f1_score', result_mlp['f1_score'])
    mlflow.log_metric('roc_auc', result_mlp['roc_auc'])
    mlflow.log_metric('best_val_loss', best_val_loss)
    
    # Log modelo
    mlflow.pytorch.log_model(model, 'mlp_model')
    
    print('[OK] MLP registrada no MLflow!')
    print(f'     ROC-AUC: {result_mlp["roc_auc"]}')
    print(f'     F1-score: {result_mlp["f1_score"]}')

## 11. Salvar Modelo MLP

In [ ]:
# Salvar modelo PyTorch
os.makedirs('../models', exist_ok=True)

# Salvar state_dict (recomendado pelo PyTorch)
model_path = '../models/mlp_model.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': INPUT_DIM,
    'architecture': 'input->64->32->1',
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'metrics': result_mlp,
    'seed': SEED
}, model_path)

print(f'[OK] Modelo MLP salvo em: {model_path}')
print(f'[OK] Metricas incluidas no arquivo')

# Salvar comparacao completa
df_all.to_csv('../models/all_results.csv')
print(f'[OK] Comparacao completa salva em: models/all_results.csv')

## 12. Conclusões

### Resultados
- MLP treinada com sucesso usando PyTorch
- Early stopping evitou overfitting, parando no ponto ideal
- Comparação completa com baselines realizada
- Todos os experimentos registrados no MLflow

### Análise de Trade-off
- **Falso Negativo** (não detectar churn) tem **maior custo de negócio** — perda do cliente
- **Falso Positivo** (alertar sem necessidade) tem custo moderado — ação de retenção desnecessária
- **Recall é métrica crítica** para esse problema
- Threshold pode ser ajustado conforme estratégia de retenção da empresa

### Próximos Passos
1. Refatorar código em `src/`
2. Criar API com FastAPI (`/predict` e `/health`)
3. Implementar testes automatizados
4. Documentação final (README, Model Card, vídeo STAR)